<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member2_risk_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install GPU libraries
!pip -q install cudf-cu12 dask-cudf-cu12 cupy-cuda12x google-cloud-bigquery

In [2]:
import cudf
import cupy as cp
import pandas as pd
import numpy as np

print("cuDF version:", cudf.__version__)
print("GPU Ready ✅")

cuDF version: 26.02.01
GPU Ready ✅


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

PROJECT_ID = "dengue-early-warning"

print("Project:", PROJECT_ID)

Project: dengue-early-warning


In [6]:
from google.colab import auth
auth.authenticate_user()

print("Authenticated ✅")

Authenticated ✅


In [7]:
from google.cloud import bigquery

PROJECT_ID = "dengue-early-warning"

client = bigquery.Client(project=PROJECT_ID)

print("Connected to BigQuery ✅")

Connected to BigQuery ✅


In [10]:
query = """
SELECT table_name
FROM `dengue-early-warning.dengue_ew.INFORMATION_SCHEMA.TABLES`
"""

tables = client.query(query).to_dataframe()

tables

,table_name
0,telemetry_daily
1,features


In [11]:
from google.cloud import bigquery
import cudf

query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.features`
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)

print("Rows:", len(df))
print("Columns:", list(df.columns))

df.head()

Rows: 306066
Columns: ['cell_id', 'lat', 'lon', 'date', 'case_count', 'rain_mm', 'case_density_14d', 'rain_lag_7to14d', 'recurrence']


,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence
0,1.25775_103.9275,1.25775,103.92750,2026-01-12,0.0,5.519703,0.0,NaN,0.0
1,1.25775_103.92975,1.25775,103.92975,2026-01-12,0.0,5.519703,0.0,NaN,0.0
2,1.26225_103.9185,1.26225,103.91850,2026-01-12,0.0,5.519703,0.0,NaN,0.0
3,1.26225_103.92075,1.26225,103.92075,2026-01-12,0.0,5.519703,0.0,NaN,0.0
4,1.26225_103.92525,1.26225,103.92525,2026-01-12,0.0,5.519703,0.0,NaN,0.0


In [12]:
gdf = cudf.from_pandas(df)

print(gdf.head())
print()
print("GPU Rows:", len(gdf))

             cell_id      lat        lon       date  case_count   rain_mm  \
0   1.25775_103.9275  1.25775  103.92750 2026-01-12         0.0  5.519703   
1  1.25775_103.92975  1.25775  103.92975 2026-01-12         0.0  5.519703   
2   1.26225_103.9185  1.26225  103.91850 2026-01-12         0.0  5.519703   
3  1.26225_103.92075  1.26225  103.92075 2026-01-12         0.0  5.519703   
4  1.26225_103.92525  1.26225  103.92525 2026-01-12         0.0  5.519703   

   case_density_14d rain_lag_7to14d  recurrence  
0               0.0            <NA>         0.0  
1               0.0            <NA>         0.0  
2               0.0            <NA>         0.0  
3               0.0            <NA>         0.0  
4               0.0            <NA>         0.0  

GPU Rows: 306066


In [13]:
gdf["rain_lag_7to14d"] = gdf["rain_lag_7to14d"].fillna(0)
gdf["recurrence"] = gdf["recurrence"].fillna(0)

print(gdf.isna().sum())

cell_id             0
lat                 0
lon                 0
date                0
case_count          0
rain_mm             0
case_density_14d    0
rain_lag_7to14d     0
recurrence          0
dtype: int64


In [14]:
gdf["risk_score"] = (
    gdf["case_density_14d"] * 0.60 +
    gdf["rain_lag_7to14d"] * 0.25 +
    gdf["recurrence"] * 0.15
)

gdf["risk_score"] = gdf["risk_score"].round(2)

gdf[["case_density_14d",
     "rain_lag_7to14d",
     "recurrence",
     "risk_score"]].head()

,case_density_14d,rain_lag_7to14d,recurrence,risk_score
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0


In [15]:
risk = (
    gdf
    .sort_values("risk_score", ascending=False)
    .reset_index(drop=True)
)

risk["rank"] = risk.index + 1

risk.head(20)

,cell_id,lat,lon,date,case_count,rain_mm,case_density_14d,rain_lag_7to14d,recurrence,risk_score,rank
0,1.33875_103.86,1.33875,103.86000,2026-06-08,0.0,14.698628,37.0,9.960594,0.240260,24.73,1
1,1.33875_103.86,1.33875,103.86000,2026-06-07,16.0,23.204172,37.0,9.512096,0.241830,24.61,2
2,1.33875_103.85775,1.33875,103.85775,2026-02-11,15.0,17.674009,31.0,16.446369,0.837838,22.84,3
3,1.33875_103.85775,1.33875,103.85775,2026-02-19,0.0,18.876726,30.0,15.708470,1.022222,22.08,4
4,1.33875_103.85775,1.33875,103.85775,2026-02-21,0.0,10.667523,30.0,15.080422,0.978723,21.92,5
5,1.33875_103.85775,1.33875,103.85775,2026-02-20,0.0,4.313012,30.0,14.908410,1.000000,21.88,6
6,1.33875_103.85775,1.33875,103.85775,2026-02-18,15.0,20.771386,30.0,14.764168,1.045455,21.85,7
7,1.33875_103.85775,1.33875,103.85775,2026-02-24,0.0,7.309486,30.0,14.599906,0.920000,21.79,8
8,1.33875_103.85775,1.33875,103.85775,2026-02-23,0.0,12.820883,30.0,14.504386,0.938776,21.77,9
9,1.33875_103.85775,1.33875,103.85775,2026-02-22,0.0,10.159543,30.0,14.365343,0.958333,21.74,10


In [17]:
import cupy as cp

case = gdf["case_density_14d"]
rain = gdf["rain_lag_7to14d"]
rec = gdf["recurrence"]

conditions = cp.argmax(
    cp.stack([
        case.values,
        rain.values,
        rec.values
    ]),
    axis=0
)

mapping = {
    0: "Case Density",
    1: "Rainfall",
    2: "Recurrence"
}

top_factor = [mapping[int(i)] for i in cp.asnumpy(conditions)]

risk["top_factor"] = top_factor

risk[["rank", "risk_score", "top_factor"]].head(10)


,rank,risk_score,top_factor
0,1,24.73,Case Density
1,2,24.61,Case Density
2,3,22.84,Case Density
3,4,22.08,Case Density
4,5,21.92,Case Density
5,6,21.88,Case Density
6,7,21.85,Case Density
7,8,21.79,Case Density
8,9,21.77,Case Density
9,10,21.74,Case Density


In [18]:
inspection = risk[
    [
        "rank",
        "cell_id",
        "lat",
        "lon",
        "date",
        "risk_score",
        "top_factor"
    ]
]

inspection.head(20)

,rank,cell_id,lat,lon,date,risk_score,top_factor
0,1,1.33875_103.86,1.33875,103.86000,2026-06-08,24.73,Case Density
1,2,1.33875_103.86,1.33875,103.86000,2026-06-07,24.61,Case Density
2,3,1.33875_103.85775,1.33875,103.85775,2026-02-11,22.84,Case Density
3,4,1.33875_103.85775,1.33875,103.85775,2026-02-19,22.08,Case Density
4,5,1.33875_103.85775,1.33875,103.85775,2026-02-21,21.92,Case Density
5,6,1.33875_103.85775,1.33875,103.85775,2026-02-20,21.88,Case Density
6,7,1.33875_103.85775,1.33875,103.85775,2026-02-18,21.85,Case Density
7,8,1.33875_103.85775,1.33875,103.85775,2026-02-24,21.79,Case Density
8,9,1.33875_103.85775,1.33875,103.85775,2026-02-23,21.77,Case Density
9,10,1.33875_103.85775,1.33875,103.85775,2026-02-22,21.74,Case Density


In [19]:
inspection.to_csv("inspection_priority.csv", index=False)

print("✅ inspection_priority.csv saved")

✅ inspection_priority.csv saved


In [23]:
inspection_pd = inspection.to_pandas()

print(type(inspection_pd))
inspection_pd.head()

<class 'pandas.core.frame.DataFrame'>


,rank,cell_id,lat,lon,date,risk_score,top_factor
0,1,1.33875_103.86,1.33875,103.86000,2026-06-08,24.73,Case Density
1,2,1.33875_103.86,1.33875,103.86000,2026-06-07,24.61,Case Density
2,3,1.33875_103.85775,1.33875,103.85775,2026-02-11,22.84,Case Density
3,4,1.33875_103.85775,1.33875,103.85775,2026-02-19,22.08,Case Density
4,5,1.33875_103.85775,1.33875,103.85775,2026-02-21,21.92,Case Density


In [27]:
query = """
SELECT table_name
FROM `dengue-early-warning.dengue_ew.INFORMATION_SCHEMA.TABLES`
"""

client.query(query).to_dataframe(create_bqstorage_client=False)

,table_name
0,telemetry_daily
1,features


In [28]:
query = """
SELECT table_name
FROM `dengue-early-warning.dengue_ew.INFORMATION_SCHEMA.TABLES`
"""

client.query(query).to_dataframe(create_bqstorage_client=False)


,table_name
0,telemetry_daily
1,features


In [29]:
table_id = "dengue-early-warning.dengue_ew.inspection_priority"

job = client.load_table_from_dataframe(
    inspection_pd,
    table_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
    ),
)

job.result()

print("✅ inspection_priority uploaded successfully")

✅ inspection_priority uploaded successfully


In [30]:
query = """
SELECT table_name
FROM `dengue-early-warning.dengue_ew.INFORMATION_SCHEMA.TABLES`
"""

client.query(query).to_dataframe(create_bqstorage_client=False)

,table_name
0,telemetry_daily
1,features
2,inspection_priority
